In [4]:
import os
import numpy as np
import pandas as pd
from ase.io import read
from mace.calculators import MACECalculator
from ase import Atoms

# =========================
# PATHS
# =========================

# ---- it0 ----
IT0_MODEL = "/home/mehuldarak/athena/mace_fps_training/mace_fps_split17_SaveIT0.model"
IT0_TRAIN = "/home/mehuldarak/athena/fps_dftfe_and_emb_data/split_17/fps_dftfe_split_17_train.extxyz"
IT0_VAL   = "/home/mehuldarak/athena/fps_dftfe_and_emb_data/split_17/fps_dftfe_split_17_val.extxyz"
IT0_TEST  = "/home/mehuldarak/athena/fps_dftfe_and_emb_data/superseed_fps_dftfe_data_centered.extxyz"

# ---- it1 ----
IT1_MODEL = "/home/mehuldarak/athena/mace_fps_training/checkpoints/mace_fps_split17_SaveIT0_256_candidate3.model"
IT1_TRAIN = "/home/mehuldarak/athena/fps_dftfe_and_emb_data/split_17_save256/fps_dftfe_split_17_train.extxyz"
IT1_VAL   = "/home/mehuldarak/athena/fps_dftfe_and_emb_data/split_17_save256/fps_dftfe_split_17_val.extxyz"

OUTPUT_EXCEL = "it0_vs_it1_force_energy_comparison.xlsx"

# =========================
# DFT E0s
# =========================
DFT_E0s = {
    3:  -190.7590256408,
    8:  -442.9888796243,
    40: -1380.1817128081,
    57: -958.0774205521
}

# =========================
# HELPERS
# =========================
def extract_filename(atoms):
    src = atoms.info.get("source_file", "")
    if not src:
        return "unknown"
    src = os.path.normpath(src)
    return os.path.basename(os.path.dirname(src))

def force_rmse(f_dft, f_mace):
    return np.sqrt(np.mean((f_mace - f_dft)**2)) * 1000  # meV/Å

def cohesive_energy(total_energy, numbers, E0_dict):
    return total_energy - sum(E0_dict[Z] for Z in numbers)

# =========================
# LOAD MODELS
# =========================
calc_it0 = MACECalculator(model_path=IT0_MODEL, device="cuda")
calc_it1 = MACECalculator(model_path=IT1_MODEL, device="cuda")

# =========================
# COMPUTE MACE E0s
# =========================
def compute_mace_E0s(calc):
    mace_E0s = {}
    for Z in DFT_E0s.keys():
        atom = Atoms(numbers=[Z], positions=[[10,10,10]],
                     cell=[20,20,20], pbc=[0,0,0])
        atom.calc = calc
        mace_E0s[Z] = atom.get_potential_energy()
    return mace_E0s

MACE_E0s_it0 = compute_mace_E0s(calc_it0)
MACE_E0s_it1 = compute_mace_E0s(calc_it1)

print("MACE E0s it0:", MACE_E0s_it0)
print("MACE E0s it1:", MACE_E0s_it1)

# =========================
# LOAD ALL STRUCTURES
# =========================
it0_structs = read(IT0_TRAIN, ":") + read(IT0_VAL, ":") + read(IT0_TEST, ":")
it1_structs = read(IT1_TRAIN, ":") + read(IT1_VAL, ":")

# Unique structure dictionary
struct_dict = {}
for atoms in it0_structs + it1_structs:
    name = extract_filename(atoms)
    struct_dict[name] = atoms

print(f"Total unique structures: {len(struct_dict)}")

# =========================
# MAIN LOOP
# =========================
rows = []

for i, (name, atoms) in enumerate(struct_dict.items()):

    print(f"[{i+1}/{len(struct_dict)}] Processing: {name}")

    numbers = atoms.get_atomic_numbers()
    n_atoms = len(numbers)

    f_dft = atoms.arrays["REF_forces"]
    dft_total = atoms.info["REF_energy"]
    dft_coh = cohesive_energy(dft_total, numbers, DFT_E0s)

    # ---- it0 ----
    atoms.calc = calc_it0
    f0 = atoms.get_forces()
    e0 = atoms.get_potential_energy()
    coh0 = cohesive_energy(e0, numbers, MACE_E0s_it0)

    rmse0 = force_rmse(f_dft, f0)
    e_err0 = (coh0 - dft_coh) / n_atoms * 1000  # meV/atom

    # ---- it1 ----
    atoms.calc = calc_it1
    f1 = atoms.get_forces()
    e1 = atoms.get_potential_energy()
    coh1 = cohesive_energy(e1, numbers, MACE_E0s_it1)

    rmse1 = force_rmse(f_dft, f1)
    e_err1 = (coh1 - dft_coh) / n_atoms * 1000  # meV/atom

    rows.append({
        "structure": name,
        "n_atoms": n_atoms,

        # forces
        "it0_force_rmse_meV_A": rmse0,
        "it1_force_rmse_meV_A": rmse1,
        "delta_force_rmse": rmse1 - rmse0,

        # energies (per atom)
        "dft_coh_eV_per_atom": dft_coh / n_atoms,
        "it0_coh_eV_per_atom": coh0 / n_atoms,
        "it1_coh_eV_per_atom": coh1 / n_atoms,

        "it0_energy_err_meV_per_atom": e_err0,
        "it1_energy_err_meV_per_atom": e_err1,
        "delta_energy_meV_per_atom": e_err1 - e_err0,

        # winners
        "better_force_model": "it0" if rmse0 < rmse1 else "it1",
        "better_energy_model": "it0" if abs(e_err0) < abs(e_err1) else "it1"
    })

df = pd.DataFrame(rows)

# =========================
# COLORING
# =========================
def highlight(row):
    if row["it0_force_rmse_meV_A"] < row["it1_force_rmse_meV_A"]:
        return ["background-color: lightgreen"] * len(row)
    else:
        return ["background-color: salmon"] * len(row)

styled = df.style.apply(highlight, axis=1)

# =========================
# SAVE EXCEL
# =========================
styled.to_excel(OUTPUT_EXCEL, index=False)

print(f"\nSaved Excel -> {OUTPUT_EXCEL}")

/home/mehuldarak/anaconda3/envs/mace310/lib/python3.10/site-packages/mace/calculators/mace.py:207: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/mehuldarak/anaconda3/envs/mace310/lib/python3.10/site-packages/mace/calculators/mace.py:207: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


MACE E0s it0: {3: -190.759033203125, 8: -442.9888916015625, 40: -1380.1817626953125, 57: -958.077392578125}
MACE E0s it1: {3: -190.759033203125, 8: -442.9888916015625, 40: -1380.1817626953125, 57: -958.077392578125}
Total unique structures: 24
[1/24] Processing: Li_110_slab__LLZO_010_La_order0_off_bestgap_2.00A_r_T1100K_4775
[2/24] Processing: Li_100_slab__LLZO_110_Li_order17_off_bestgap_2.50A_r_T550K_2600
[3/24] Processing: completed_Li_110_slab__LLZO_001_Zr_code93_sto_bestgap_2.50A_r_T550K_3675
[4/24] Processing: completed_Li_100_slab__LLZO_001_Zr_code93_sto_bestgap_3.00A_r_T550K_125
[5/24] Processing: Li_111_slab__LLZO_110_Li_order17_off_bestgap_2.00A_r_T1100K_3425
[6/24] Processing: completed_Li_100_slab__LLZO_010_La_order0_off_bestgap_2.50A_r_T1100K_2200
[7/24] Processing: completed_Li_100_slab__LLZO_010_La_order0_off_bestgap_2.50A_r_T1100K_3800
[8/24] Processing: Li_111_slab__LLZO_011_La_code71_sto_bestgap_2.50A_r_T550K_125
[9/24] Processing: Li_100_slab__LLZO_001_Zr_code93_sto_b